In [4]:
import glob
import os
import random
from typing import List, Optional, Tuple

import pandas as pd
from IPython.core.display import Image as IPyImage
from rdkit import Chem, rdBase
from rdkit.Chem import Draw

# Suppress RDKit warnings (optional, can be noisy)
rdBase.DisableLog("rdApp.warning")

In [7]:
INPUT_PATH_PATTERN: str = "../data/sampling/reinvent_filtered_*.csv"  # <--- CHANGE THIS TO MATCH YOUR FILE LOCATION/PATTERN

# Size (width, height) of each individual molecule image within the grid.
# This directly impacts the resolution of the final PNG. Larger is higher quality.
PNG_SUB_IMG_SIZE: Tuple[int, int] = (500, 500)

# Maximum number of molecules per row in the grid display.
MOLS_PER_ROW: int = 5

# Maximum number of molecules to display (sampling threshold)
MAX_DISPLAY_MOLS: int = 25

In [8]:
def find_smiles_column(df: pd.DataFrame) -> Optional[str]:
    """
    Finds the SMILES column in a DataFrame, checking for 'smiles' or 'SMILES'.

    Args:
        df: The pandas DataFrame to search within.

    Returns:
        The name of the SMILES column ('smiles' or 'SMILES') if found, otherwise None.
    """
    if "smiles" in df.columns:
        return "smiles"
    elif "SMILES" in df.columns:
        return "SMILES"
    else:
        return None


def smiles_to_mols(
    smiles_list: List[str],
) -> Tuple[List[Chem.Mol], List[str]]:
    """
    Converts a list of SMILES strings to RDKit Mol objects.

    Args:
        smiles_list: A list of SMILES strings.

    Returns:
        A tuple containing:
        - A list of valid RDKit Mol objects with 'Original_SMILES' property set.
        - A list of SMILES strings that failed to parse or were invalid.
    """
    mols: List[Chem.Mol] = []
    invalid_smiles: List[str] = []
    for i, smi in enumerate(smiles_list):
        # Ensure input is a non-empty string before processing
        if not isinstance(smi, str) or not smi.strip():
            invalid_smiles.append(str(smi)) # Record the problematic input
            continue
        mol = Chem.MolFromSmiles(smi)
        if mol:
            # Store original SMILES and an index as properties
            mol.SetProp("_Name", f"Mol_{i}")
            mol.SetProp("Original_SMILES", smi)
            mols.append(mol)
        else:
            invalid_smiles.append(smi)
    return mols, invalid_smiles


def generate_and_save_png_grid(
    mols: List[Chem.Mol],
    output_filepath: str, # Full path including .png extension
    legends: Optional[List[str]] = None,
    mols_per_row: int = 5,
    sub_img_size: Tuple[int, int] = (300, 300),
) -> Optional[str]:
    """
    Generates an RDKit molecule grid image, saves it ONLY as a PNG file,
    handling potential IPython object returns, and returns the filepath upon success.

    Args:
        mols: List of RDKit Mol objects to include in the grid.
        output_filepath: The full path where the PNG image file should be saved
                         (must end with .png).
        legends: Optional list of strings to display below each molecule.
                 Defaults to original SMILES if available.
        mols_per_row: Maximum number of molecules per row in the grid.
        sub_img_size: Tuple representing the size (width, height) of each sub-image.
                      Larger sizes result in higher resolution PNGs.

    Returns:
        - The output filepath as a string, if saving is successful.
        - None if no molecules are provided or saving fails.
    """
    if not mols:
        print("  ⚠️ No molecules provided for PNG grid generation.")
        return None

    # Ensure the output path ends with .png (case-insensitive check)
    if not output_filepath.lower().endswith(".png"):
        print(f"  ❌ Error: Output filepath '{output_filepath}' must end with .png")
        return None

    # Determine effective legends
    effective_legends = legends
    if effective_legends and len(effective_legends) != len(mols):
        print(f"  ⚠️ Legend count ({len(effective_legends)}) != mol count ({len(mols)}). Using default legends.")
        effective_legends = None # Fallback to default
    if not effective_legends:
         try:
             # Attempt to use the stored original SMILES property
             effective_legends = [m.GetProp("Original_SMILES") for m in mols]
         except KeyError:
             print("  ⚠️ Could not find 'Original_SMILES' property for default legends. Omitting legends.")
             effective_legends = [""] * len(mols) # Use blank legends if property missing

    generated_object = None
    try:
        # Generate PNG - Expecting PIL Image or IPython Image
        generated_object = Draw.MolsToGridImage(
            mols,
            molsPerRow=mols_per_row,
            subImgSize=sub_img_size, # Control resolution here
            legends=effective_legends,
            useSVG=False, # Explicitly ask for raster image type
        )

        # Check if RDKit returned an IPython.core.display.Image object
        # This seems to be the behavior in the user's environment
        if isinstance(generated_object, IPyImage) and hasattr(generated_object, 'data'):
            # The .data attribute should contain the raw PNG bytes
            png_data = generated_object.data
            if isinstance(png_data, bytes):
                 # Write the bytes data to the file in binary mode ('wb')
                 with open(output_filepath, "wb") as f:
                     f.write(png_data)
                 print(f"  ✅ PNG Image saved to: {output_filepath} (SubImgSize: {sub_img_size})")
                 return output_filepath # Return filepath string on success
            else:
                 # Should not happen if .data exists, but check anyway
                 print(f"  ❌ Error: IPython Image object's .data attribute is not bytes (Type: {type(png_data)}). Cannot save.")
                 return None
        else:
             # If it's not the expected IPython object, report error
             print(f"  ❌ Error: RDKit MolsToGridImage (PNG) did not return a recognized Image object with data attribute.")
             print(f"     Object type received: {type(generated_object)}")
             # Note: Could add a check for PIL.Image here if RDKit's behavior varies,
             # but based on errors, IPyImage seems the primary return type to handle.
             # from PIL import Image as PILImage
             # if isinstance(generated_object, PILImage.Image): ... handle PIL saving ...
             return None

    except Exception as e:
        # Catch any other errors during generation or saving
        print(f"  ❌ Error during PNG generation/saving for {output_filepath}: {type(e).__name__} - {e}")
        # import traceback # Uncomment for detailed stack trace during debugging
        # traceback.print_exc() # Uncomment for detailed stack trace during debugging
        return None


# %% [markdown]
# ## 4. Main Processing Function

# %%
def process_csv_file(filepath: str):
    """
    Reads a CSV file, processes SMILES, generates and saves a molecule grid PNG.
    **Image rendering in the notebook output is disabled.**

    Args:
        filepath: Path to the input CSV file.
    """
    filename = os.path.basename(filepath)
    output_dir = os.path.dirname(filepath)
    # Create the base filename for the output PNG
    base_filename = os.path.splitext(filename)[0]
    output_png_filename = f"{base_filename}_molecules.png"
    output_png_filepath = os.path.join(output_dir, output_png_filename)

    print(f"Processing: {filename}")

    try:
        # --- Read CSV ---
        df = pd.read_csv(filepath)
        if df.empty:
            print(f"  ⚠️ Skipping: File {filename} is empty.")
            print("-" * 30)
            return

        # --- Identify SMILES column ---
        smiles_col = find_smiles_column(df) # Assumes this function exists
        if smiles_col is None:
            print(f"  ⚠️ Skipping: No 'smiles' or 'SMILES' column found.")
            print("-" * 30)
            return

        # --- Extract and Clean SMILES strings ---
        smiles_list = df[smiles_col].dropna().astype(str).tolist()
        if not smiles_list:
            print(f"  ⚠️ Skipping: No valid SMILES strings found after cleaning.")
            print("-" * 30)
            return

        # --- Convert SMILES to RDKit Mol objects ---
        mols, invalid_smiles = smiles_to_mols(smiles_list) # Assumes this function exists

        if invalid_smiles:
            print(f"  ℹ️ Found {len(invalid_smiles)} invalid/empty SMILES strings. Examples: {invalid_smiles[:5]}")

        if not mols:
            print(f"  ❌ Skipping: Could not generate any valid molecules.")
            print("-" * 30)
            return

        num_mols = len(mols)
        print(f"  ℹ️ Found {num_mols} valid molecules.")

        # --- Decide on visualization: All or Sample ---
        mols_to_draw: List[Chem.Mol]
        # title_suffix is no longer used for display but kept for context if needed
        title_suffix: str

        if num_mols <= MAX_DISPLAY_MOLS: # Assumes MAX_DISPLAY_MOLS is defined
            mols_to_draw = mols
            title_suffix = f"(Using all {num_mols})"
        else:
            print(f"  ℹ️ Sampling {MAX_DISPLAY_MOLS} molecules randomly from {num_mols}.")
            mols_to_draw = random.sample(mols, MAX_DISPLAY_MOLS)
            title_suffix = f"(Using random {MAX_DISPLAY_MOLS} of {num_mols})"

        # --- Prepare Legends ---
        try:
            legends = [m.GetProp("Original_SMILES") for m in mols_to_draw]
        except KeyError:
            legends = None

        # --- Generate Grid Image and Save (PNG only) ---
        # Assumes generate_and_save_png_grid function exists and handles saving
        # Assumes MOLS_PER_ROW and PNG_SUB_IMG_SIZE configuration variables exist
        png_filepath_or_none = generate_and_save_png_grid(
            mols=mols_to_draw,
            output_filepath=output_png_filepath,
            legends=legends,
            mols_per_row=MOLS_PER_ROW,
            sub_img_size=PNG_SUB_IMG_SIZE
        )

        # --- Image Display Section Removed ---
        if not png_filepath_or_none:
             # This message is printed if generate_and_save_png_grid returned None
             print(f"  ⚠️ PNG image generation/saving failed for {filename}.")
        # No 'else' block needed as we are not displaying upon success.

    except FileNotFoundError:
        print(f"  ❌ Error: File not found at {filepath}")
    except pd.errors.EmptyDataError:
        print(f"  ⚠️ Skipping: File {filename} is empty.")
    except pd.errors.ParserError:
        print(f"  ❌ Error: Could not parse CSV file {filename}. Check format.")
    except Exception as e:
        print(f"  ❌ An unexpected error occurred while processing {filename}: {type(e).__name__} - {e}")
        # import traceback # Uncomment for full traceback during debugging
        # traceback.print_exc() # Uncomment for full traceback during debugging
    finally:
        # Separator for clarity between file processing outputs
        print("-" * 30)


# %% [markdown]
# ## 5. Run Processing Loop

# %%
# --- Configuration Check ---
# Ensure INPUT_PATH_PATTERN is defined before use
if 'INPUT_PATH_PATTERN' not in globals():
    print("❌ Configuration Error: INPUT_PATH_PATTERN is not defined.")
    INPUT_PATH_PATTERN = "*.csv" # Provide a default to potentially avoid crash

# --- Find Files ---
print(f"Searching for files matching pattern: '{INPUT_PATH_PATTERN}'")
# Use recursive=True to support '**' pattern for searching subdirectories
file_list: List[str] = glob.glob(INPUT_PATH_PATTERN, recursive=True)

if not file_list:
    print(f"❌ No files found matching pattern: '{INPUT_PATH_PATTERN}'")
    print("   Please check the INPUT_PATH_PATTERN variable in section 2.")
else:
    print(f"✅ Found {len(file_list)} file(s):")
    # Sort for consistent processing order
    file_list.sort()
    for f_path in file_list:
        print(f"  - {f_path}") # Show full path found by glob
    print("-" * 30)

    # --- Process Each File ---
    # Ensure the processing function is available
    if 'process_csv_file' not in globals() or not callable(process_csv_file):
         print("❌ Execution Error: 'process_csv_file' function not defined or not callable.")
         print("   Please ensure the function definition in Cell 4 has been executed.")
    else:
        for filepath in file_list:
            process_csv_file(filepath) # Call the main processing function for each file

    print("\nProcessing finished.")


# %% [markdown]
# ## 6. End of Script

Searching for files matching pattern: '../data/sampling/reinvent_filtered_*.csv'
✅ Found 2 file(s):
  - ../data/sampling/reinvent_filtered_no_cs.csv
  - ../data/sampling/reinvent_filtered_with_cs.csv
------------------------------
Processing: reinvent_filtered_no_cs.csv
  ℹ️ Found 9 valid molecules.
  ✅ PNG Image saved to: ../data/sampling/reinvent_filtered_no_cs_molecules.png (SubImgSize: (500, 500))
------------------------------
Processing: reinvent_filtered_with_cs.csv
  ℹ️ Found 7805 valid molecules.
  ℹ️ Sampling 25 molecules randomly from 7805.
  ✅ PNG Image saved to: ../data/sampling/reinvent_filtered_with_cs_molecules.png (SubImgSize: (500, 500))
------------------------------

Processing finished.
